[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ivanvykopal/nlp-kinit-2026/blob/main/examples/sft/fft.ipynb)

# Full Fine-Tuning (FFT) on Tower of Hanoi

This notebook trains **every parameter** of a 135M model to imitate
worked solutions of the Tower of Hanoi puzzle. The answers it imitates
are not all correct: one in five of the training answers has a single
move deliberately broken — swapped, replaced, deleted or inserted. That
is not a mistake in the dataset; it is there so you can watch what
supervised training does with imperfect demonstrations.

What it does is imitate them. Train a model to reproduce answers that are
wrong one time in five and it reproduces that error rate too — and a
31-move solution only counts if every single move is legal. The GSPO
notebook shows the alternative: score the model's own attempts with a
checker instead of copying someone else's answers.

Runtime: ~10 minutes on a free Colab T4 (about eight of which is the
training cell; the rest is setup and evaluation).

## 0. Setup

Install the dependencies (minimum versions these notebooks were validated against).

In [ ]:
get_ipython().system('pip install -q "transformers>=4.56,<5" "trl>=0.24" "peft>=0.14" "datasets>=3.0" accelerate matplotlib pandas')

In [ ]:
get_ipython().system('git clone --depth 1 https://github.com/<your-username>/nlp-kinit-2026')
import sys; sys.path.insert(0, "nlp-kinit-2026")

from lab import evaluation, generation, plotting, report

## Shared code

The cells below define the constants and the Tower of Hanoi
task logic (`tasks/hanoi.py`) directly in the notebook's own namespace —
run them once from the top; everything after uses these names without any
`import`. Generic infrastructure (batched generation, evaluation,
reporting, plotting) is *imported for real* in the cell above, from
`lab/`.


In [ ]:
"""Generic constants shared by every notebook: model, dirs, token/training budgets.

No task-shaped constants here — those live in the task module (see
`tasks/hanoi.py`'s `PROBE_GROUP_VALUES`).
"""
from pathlib import Path

MODEL_NAME = "HuggingFaceTB/SmolLM2-135M-Instruct"  # We will be working with a small HF model
SEED = 42

Every notebook reads the same dataset from the HuggingFace Hub. It has five
named splits: `train` (answers to imitate, one in five deliberately wrong),
`grpo_train` (puzzles only, no answers — the split name predates our rename
to GSPO), `heldout` (the number we care about), `train_instances` (a
memorisation check) and `extrapolation` (bigger puzzles than any in
training).

In [ ]:
DATASET_REPO = "ivykopal/nlp-kinit-2026-hanoi"

Notebooks run from different working directories — `examples/rl/` locally,
`/content` on Colab — so a bare `Path("results")` would point somewhere
different in each one, and the GSPO notebook would fail to find the
checkpoint the FFT notebook saved. This walks up from the current directory
to find the one folder holding both `lab/` and `config.py`.

In [ ]:
def _repo_root() -> Path:
    import sys
    candidates = [Path.cwd(), *Path.cwd().parents]
    candidates += [Path(p) for p in sys.path if p]
    for c in candidates:
        if c.is_dir() and (c / "lab").is_dir() and (c / "config.py").is_file():
            return c
    return Path.cwd()  # fallback: original <cwd>/results behaviour


RESULTS_DIR = _repo_root() / "results"

How many tokens we let the model read and write. `MAX_SEQ_LENGTH` caps
training examples; `EVAL_MAX_NEW_TOKENS` has to be generous enough for a
31-move answer, or a correct solution gets cut off and scored as a failure.

In [ ]:
MAX_SEQ_LENGTH = 384
EVAL_MAX_NEW_TOKENS = 576
EVAL_BATCH_SIZE = 16

### BUDGETS: supervised fine-tuning

FFT and LoRA train on the same schedule and only differ in learning rate —
LoRA's update is confined to a low-rank subspace, so it needs a larger step
size to move as far as full fine-tuning does in the same number of steps.

In [ ]:
SFT_MAX_STEPS = 300
SFT_BATCH_SIZE = 4
SFT_GRAD_ACCUM = 4
FFT_LEARNING_RATE = 5e-5

## Task: Tower of Hanoi

This module is the one place that knows what "Tower of Hanoi" means:
how to parse a model's answer, how to check whether a sequence of moves
actually solves the puzzle, how to turn a dataset row into a training
example, and how to reward a rollout. Everything in `lab/` is generic —
it never mentions a peg or a disk by name — and depends only on the
functions below.

Swapping this tutorial to a different verifiable task (Sudoku, a graph
coloring problem, anything with a checker) means writing a new module
shaped like this one; `lab/` would not change at all.

A move is a plain `(source, target)` tuple of peg names — `("A", "C")`
reads as "move the top disk of A onto C". No `Move` class: a tuple is
already immutable, printable, and comparable, and one regex below is the
entire move syntax.

In [ ]:
"""Tower of Hanoi: parsing, solving, replaying, scoring, and reward."""
from __future__ import annotations

import random
import re
from typing import List

PEG_NAMES = ["A", "B", "C"]

# The whole move syntax: `A->C`, with optional spaces around the arrow.
MOVE_RE = re.compile(r"([{pegs}])\s*->\s*([{pegs}])".format(pegs="".join(PEG_NAMES)))

Two tiny helpers turn a move tuple into the text form the model reads and
writes, and back — everything else in this module works with `(source,
target)` tuples, so the text form only exists at the model boundary.

In [ ]:
def move_to_text(move) -> str:
    return "{}->{}".format(*move)


def moves_to_text(moves) -> str:
    return "\n".join(move_to_text(m) for m in moves)

Turning model output back into moves needs two different levels of
strictness. `parse_output` is used for scoring a whole flat solution: it
walks every line and keeps a strict per-line count, so junk is counted
rather than silently dropped. `first_move_in` is used by the per-step
formulation, where the model is asked for exactly one move: it just finds
the first well-formed move anywhere in the text and ignores the rest.

In [ ]:
def parse_output(text: str):
    """Parse a whole completion into moves.

    Returns `(moves, n_lines, n_unparsed)`: every non-empty line is either a
    move or counted as unparseable — junk is counted, never silently
    dropped, so the caller can tell a clean list of moves from noisy output.
    """
    moves, n_lines, n_unparsed = [], 0, 0
    for line in text.splitlines():
        line = line.strip()
        if not line:
            continue
        n_lines += 1
        match = MOVE_RE.fullmatch(line)
        if match:
            moves.append(match.groups())
        else:
            n_unparsed += 1
    return moves, n_lines, n_unparsed

A reference solver, used two ways below: to build the optimal
demonstrations the SFT notebooks train on, and to know the optimal move
count for scoring — `2**n - 1`, the closed form for this recursion.

In [ ]:
def solve_hanoi(n_disks, source, auxiliary, target) -> List[tuple]:
    """The unique optimal solution: move n-1 aside, move the bottom disk, move them back."""
    if n_disks == 0:
        return []
    return (
        solve_hanoi(n_disks - 1, source, target, auxiliary)
        + [(source, target)]
        + solve_hanoi(n_disks - 1, auxiliary, source, target)
    )


def optimal_move_count(n_disks: int) -> int:
    return 2 ** n_disks - 1

### The verifier

A board is just a `dict` of `{peg: [disks]}`, top disk last in the list; `apply_move`
is the single stepping primitive (used by the flat replay below); and `replay` folds it over a list
of moves. An illegal move ends the attempt — whatever follows it is not
played, so it cannot count for anything.

In [ ]:
def new_pegs(n_disks, source, auxiliary, target) -> dict:
    """All disks stacked on `source`; the other two pegs start empty."""
    return {source: list(range(n_disks, 0, -1)), auxiliary: [], target: []}


def is_legal(pegs: dict, move) -> bool:
    """Legal iff it moves the top disk of one peg onto a bigger (or empty) peg."""
    source, target = move
    if source == target or not pegs[source]:
        return False
    return not pegs[target] or pegs[source][-1] < pegs[target][-1]

`apply_move` is the single stepping primitive — used by the flat `replay` below.

In [ ]:
def apply_move(pegs: dict, move) -> "tuple[dict, bool]":
    """Legality-checked step. Returns (new_pegs, legal); `pegs` is never mutated."""
    if not is_legal(pegs, move):
        return pegs, False
    source, target = move
    stepped = {peg: list(stack) for peg, stack in pegs.items()}
    stepped[target].append(stepped[source].pop())
    return stepped, True


def is_solved(pegs: dict, n_disks: int, target: str) -> bool:
    return len(pegs[target]) == n_disks

`replay` folds `apply_move` over a whole list of moves, for scoring a flat
solution. An illegal move ends the attempt — whatever follows it is not
played, so it cannot count for anything.

In [ ]:
def replay(moves, n_disks, source, auxiliary, target) -> "tuple[dict, bool]":
    """Play `moves` from the start, stopping at the first illegal move.

    Returns the board it reached and whether an illegal move ended it.
    """
    pegs = new_pegs(n_disks, source, auxiliary, target)
    for move in moves:
        pegs, legal = apply_move(pegs, move)
        if not legal:
            return pegs, True
    return pegs, False

### Loading the dataset

The dataset lives on the HuggingFace Hub as five named splits — `train`,
`grpo_train`, `heldout`, `train_instances`, `extrapolation`. One row looks
like this:

```json
{
  "prompt": "...", "n_disks": 3, "source": "A", "auxiliary": "B", "target": "C",
  "target_response": "...", "optimal_response": "...",
  "corrupted": false, "corruption": null
}
```

In [ ]:
def load_split(name: str, repo_id: str):
    """Load one named split of the dataset from the HuggingFace Hub.

    Returns a `datasets.Dataset` — already iterable/indexable like a list of
    dicts (for `sum(r["corrupted"] for r in rows)`-style inspection) and
    already has `.map()` (for building the SFT/GSPO training format), so
    there's no separate "plain rows" vs. "Dataset" loading step.
    """
    from datasets import load_dataset

    return load_dataset(repo_id, split=name)

### Grouping for the by-group breakdown and the mid-training probe

Hanoi's natural grouping is disk count. A different task might group by
difficulty, grid size, or not at all — `lab.evaluation`/`lab.probe` accept
`group_key=None` and simply skip the breakdown when a task has nothing to
group by.

In [ ]:
def PROBE_GROUP_KEY(sample: dict) -> int:
    return sample["n_disks"]


PROBE_GROUP_VALUES = [4]

### `compute_stats`: what a model's answer scores as

`compute_stats` is what evaluation reports, and it deliberately reports
only three things: **did it solve the puzzle**, **how many moves it wrote
against the 2**n - 1 optimum**, and **was there anything in the output
that was not a move**. Nothing else — every extra field is one more column
to explain and one more thing that can quietly disagree with `solved`.

`"solved"` (a bool) is the only field `lab/` requires; the rest is
free-form, and `lab.evaluation.aggregate` averages whatever numeric/bool
fields it finds without needing to know their names.

In [ ]:
def compute_stats(prediction: str, sample: dict) -> dict:
    """Replay one completion and report what evaluation shows."""
    moves, n_lines, n_unparsed = parse_output(prediction)
    pegs, _illegal = replay(moves, sample["n_disks"], sample["source"],
                            sample["auxiliary"], sample["target"])
    return {
        "solved": is_solved(pegs, sample["n_disks"], sample["target"]),
        "total_moves": len(moves),
        "optimal_moves": optimal_move_count(sample["n_disks"]),
        "unparsed": n_unparsed > 0,
    }

### From dataset row to training example

Three small functions turn one dataset row into what a trainer expects: a
prompt, and — for the two supervised methods — the answer to train on.

In [ ]:
DEFAULT_SYSTEM_PROMPT = (
    "You are an expert algorithmic problem solver. "
    "Solve the Tower of Hanoi puzzle optimally. "
    "Return ONLY one move per line in the format 'A->C'. "
    "Do not provide any explanation."
)

`to_chat_prompt` wraps `DEFAULT_SYSTEM_PROMPT` and the puzzle's own prompt
text into the system+user half of a conversation. `to_sft_format` adds the
assistant turn FFT/LoRA train on.

In [ ]:
def to_chat_prompt(sample: dict) -> List[dict]:
    """The prompt half of the conversation (system + user)."""
    return [
        {"role": "system", "content": DEFAULT_SYSTEM_PROMPT},
        {"role": "user", "content": sample["prompt"]},
    ]


def to_sft_format(sample: dict) -> dict:
    """Prompt/completion format for SFTTrainer (loss on the answer only)."""
    return {
        "prompt": to_chat_prompt(sample),
        "completion": [{"role": "assistant", "content": sample["target_response"]}],
    }

## What fine-tuning actually changes

The model we just downloaded is good at exactly
one thing: given some text, guess which token comes next. Nobody taught
it Tower of Hanoi. Ask it for a solution and you get something that
*looks* like an answer — confident, fluent, wrong.

**Fine-tuning** is how we fix that, and it is worth being precise about
what it does and does not do. It does not add a new skill from nothing.
It shifts *which* continuations the model finds likely, by showing it
examples of the input/output pairs we want. Supervised fine-tuning (SFT)
is the plainest version: collect prompts with the answers you wish the
model had given, and train it to give those answers.

Concretely, we minimise the negative log-likelihood of the answer tokens:

$$
\mathcal{L}(\theta) = -\sum_{t} \log p_\theta\big(y_t \mid x,\, y_{<t}\big)
$$

Read that piece by piece. $x$ is the prompt — here, a Tower of Hanoi
puzzle. $y$ is the answer we want, and $y_t$ is its $t$-th token. $\theta$
stands for every weight in the model, the things training is allowed to
change. Each term in the sum asks one question: given the puzzle and the
answer so far, how much probability did the model put on the *correct*
next token? Training pushes those probabilities up, which is the same as
pushing the loss down.

Now notice what the sum runs over: **answer tokens only**. That is what
`completion_only_loss=True` does in the config further down. We want the
model to get better at producing answers, not at predicting puzzles we
are going to type in ourselves.

![Loss is computed on answer tokens only](https://raw.githubusercontent.com/ivanvykopal/nlp-kinit-2026/main/images/sft_loss_mask.png)

*The prompt tokens are context; only the answer tokens contribute to the
loss. (Shown as whole words for readability; a real tokenizer splits many
of them into smaller subword pieces, but the masking works the same way —
by token, not by word.)*

This objective has a ceiling built into it, and the whole tutorial hinges
on seeing that ceiling. The loss is smallest when the model reproduces
the demonstrations — *all* of them, including the bad ones. One in five
answers in our training data has a deliberately broken move, so a model
that fits this data perfectly has also learned to make broken moves at
some rate. In a 31-move solution, where one illegal move invalidates
everything after it, that rate is fatal.

There are two ways out, and this tutorial covers both: score the model's
own attempts against a verifier instead of copying answers (the GSPO
notebook), or ask for one move at a time, so a mistake cannot compound
(the per-step notebooks).

## 1. Load the shared dataset

Every notebook in this set reads the same dataset from the HuggingFace
Hub. It comes pre-split, and each split answers a different question:

| split | what it measures |
|---|---|
| `train` | what we train on — one answer in five has a broken move |
| `heldout` | puzzles never seen in training; this is the number we care about |
| `train_instances` | same disk counts, puzzles the model *did* see — a memorisation check |
| `extrapolation` | more disks than anything in training |

In [ ]:
train_rows = load_split("train", DATASET_REPO)

eval_splits = {
    "heldout": load_split("heldout", DATASET_REPO),
    "train_instances": load_split("train_instances", DATASET_REPO),
    "extrapolation": load_split("extrapolation", DATASET_REPO),
}

print(f"train: {len(train_rows)} rows, "
      f"{sum(r['corrupted'] for r in train_rows)} corrupted")
for name, rows in eval_splits.items():
    print(f"{name}: {len(rows)} instances, disks {sorted({r['n_disks'] for r in rows})}")

### A raw dataset row

Before anything else, look at what one training row actually holds: the
puzzle prompt, its coordinates, the optimal solution, and the answer we
will train on — which is the optimal one unless this row was corrupted.

In [ ]:
print(train_rows[0])

### The verifier, on a toy example

Everything in this tutorial rests on `compute_stats` being right, so check
it before trusting it on real model output. Feed it four hand-written
answers to a 2-disk puzzle — one correct, three wrong in three different
ways — and read the four verdicts.

In [ ]:
toy_sample = {"n_disks": 2, "source": "A", "auxiliary": "B", "target": "C"}

print(compute_stats("A->B\nA->C\nB->C", toy_sample))   # the optimal 2-disk solution
print(compute_stats("A->B\nB->A\nA->C", toy_sample))   # legal moves, does not solve it
print(compute_stats("A->C\nA->C", toy_sample))         # second move is illegal: attempt ends
print(compute_stats("I think A->B\nthen A->C", toy_sample))  # neither line is a bare move

### What evaluation needs

Evaluating a model on this task takes exactly two task-specific functions:
one that turns a puzzle into a chat prompt, and one that scores whatever
the model wrote. Bundling them means every evaluation call below fits on
one line.

In [ ]:
TASK = evaluation.FlatTask(compute_stats=compute_stats, to_chat_prompt=to_chat_prompt)

### What a corrupted demonstration looks like

This is the corruption from the title, made concrete. Print an optimal
solution beside the answer we actually train on and mark the lines that
differ. Everything after that marked line is still legal-looking text,
which is exactly why the model has no way to tell it is being misled.

In [ ]:
example = next(r for r in train_rows if r["corrupted"] and r["n_disks"] == 3)
print(example["prompt"])
print()
print(f"{'OPTIMAL':<14}{'SUPERVISED TARGET':<14}   corruption: {example['corruption']}")
print("-" * 45)
optimal_lines = example["optimal_response"].splitlines()
target_lines = example["target_response"].splitlines()
for i in range(max(len(optimal_lines), len(target_lines))):
    left = optimal_lines[i] if i < len(optimal_lines) else ""
    right = target_lines[i] if i < len(target_lines) else ""
    marker = "  <-- differs" if left != right else ""
    print(f"{left:<14}{right:<14}{marker}")

## 2. Model and tokenizer

If you let `dtype` default here, the run crashes on a Colab T4. SmolLM2's
config asks for `bfloat16`, and a T4 is too old to support it (it is a
Turing card, compute capability 7.5) — so we ask for `float32` explicitly.
Everything else in this cell is routine: pick a pad token if the
tokenizer has none, and switch off the KV cache, which only helps
generation and gets in the way during training.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from trl import SFTConfig, SFTTrainer

METHOD = "FFT"
OUTPUT_DIR = RESULTS_DIR / "fft_model"
set_seed(SEED)

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float32)
model.config.use_cache = False

print(f"parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")
print(f"all trainable: {sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6:.1f}M")

## 3. Baseline: the untrained instruct model

Score the model *before* training it. Without this number, any result
after training is unreadable — you cannot tell a real improvement from a
puzzle that was easy all along. Expect close to zero here: nobody has
ever shown this model a Tower of Hanoi solution.

In [ ]:
baseline = evaluation.evaluate_model(
    model, tokenizer, {"heldout": eval_splits["heldout"]}, f"{METHOD} (before training)", TASK,
    max_new_tokens=EVAL_MAX_NEW_TOKENS, batch_size=EVAL_BATCH_SIZE,
)
report.print_report(baseline)

## 4. Supervised fine-tuning

First reshape each row into the prompt/completion pair TRL expects. Two
settings in the config below are worth knowing about before we get there:

* `completion_only_loss=True` — train on the answer tokens only, not on
  the system and user turns. We want the model better at *writing*
  answers, not at predicting the puzzles we type in ourselves.
* `bf16=False, fp16=False` — set both explicitly, or the run crashes on a
  T4. TRL's `bf16` default resolves to `True` when `fp16` is left unset,
  and bf16 is exactly what a T4 cannot do.

In [ ]:
train_dataset = train_rows.map(to_sft_format, remove_columns=train_rows.column_names)
eval_dataset = eval_splits["heldout"].map(
    to_sft_format, remove_columns=eval_splits["heldout"].column_names
)

The config itself. `max_length` uses the shared `MAX_SEQ_LENGTH` budget so
that a 31-move answer fits without being cut off; `max_steps` and
`learning_rate` come from `config.py`, where the FFT and LoRA budgets are
set side by side. The rest are ordinary training mechanics, shared by
every supervised run in this notebook set.

In [ ]:
training_args = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    max_steps=SFT_MAX_STEPS,
    learning_rate=FFT_LEARNING_RATE,
    weight_decay=0.01,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    per_device_train_batch_size=SFT_BATCH_SIZE,
    per_device_eval_batch_size=SFT_BATCH_SIZE,
    gradient_accumulation_steps=SFT_GRAD_ACCUM,
    max_length=MAX_SEQ_LENGTH,
    completion_only_loss=True,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=25,
    save_strategy="no",
    bf16=False,
    fp16=False,
    report_to="none",
    seed=SEED,
)

With the dataset and the config ready, hand both to the trainer and go.
This is the cell that takes the time — about eight minutes on a T4. The
loss it prints should fall; if it is flat from the start, something above
is wrong.

In [ ]:
trainer = SFTTrainer(
    model=model, args=training_args, train_dataset=train_dataset,
    eval_dataset=eval_dataset, processing_class=tokenizer,
)
trainer.train()

Save the checkpoint.

In [ ]:
trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))
print(f"saved to {OUTPUT_DIR}")

## 5. Training curves

The loss curve is worth a moment, because it is easy to over-read. A low
loss here means "reproduces the demonstrations well" — and one in five of
those demonstrations contains a broken move. So a beautiful curve is
entirely compatible with a model that has learned to make broken moves at
some rate. Section 6 is the number that actually tells you.

In [ ]:
history = plotting.history_from_log(trainer.state.log_history)
plotting.plot_history(history, ["loss", "eval_loss"], RESULTS_DIR / "fft_loss.png",
                       title="FFT: loss", ylabel="cross-entropy")

## 6. Evaluation

Now run the trained model on all four splits and score every answer with
the verifier. Two columns to read: `solved` means every disk ended up on
the target peg, and `optimal` means solved in the minimum number of moves
with no illegal move along the way.

In [ ]:
fft_report = evaluation.evaluate_model(model, tokenizer, eval_splits, METHOD, TASK,
                                        group_key=PROBE_GROUP_KEY,
                                        max_new_tokens=EVAL_MAX_NEW_TOKENS, batch_size=EVAL_BATCH_SIZE)
report.print_report(fft_report)
report.save_report(fft_report, RESULTS_DIR)

## 7. A single prediction, in detail

Averages hide what the model is actually writing. Print one 4-disk
held-out puzzle in full — the prompt, the moves the model produced, and
the verifier's verdict on them.

In [ ]:
heldout = eval_splits["heldout"]
index = next(i for i, r in enumerate(heldout) if r["n_disks"] == 4)
report.print_example(fft_report["splits"]["heldout"]["completions"][index], heldout[index],
                      compute_stats)

## 8. Where the errors are

One aggregate number tells you how often the model failed, not *how*. So
list the unsolved held-out puzzles with the move count the model wrote
next to the count the puzzle needs. Far fewer moves means it stopped
early or hit an illegal move; far more means it rambled past the optimum.
That slip is exactly what a verifiable reward can remove.

In [ ]:
failures = [
    (compute_stats(completion, sample), sample)
    for completion, sample in zip(fft_report["splits"]["heldout"]["completions"], heldout)
]
failures = [f for f in failures if not f[0]["solved"]]

print(f"{len(failures)}/{len(heldout)} held-out instances unsolved\n")
for stats, sample in failures[:10]:
    print(
        f"  {sample['n_disks']} disks  "
        f"moves {stats['total_moves']:>3} / {stats['optimal_moves']:<3} required"
        + ("   (output had non-move lines)" if stats["unparsed"] else "")
    )